In [0]:
"""
==============================================================
Module: Superstore Silver layer orchestrator ETL orchestrator

Purpose:
    This module orchestrates the ETL pipeline from the Bronze layer to the Silver layer for multiple tables.
    It processes data from the Bronze layer and applies necessary transformations, data quality checks,
    deduplication, and updates the corresponding Silver and audit tables. It also tracks performance and
    operational metrics for each table processed.

Key Features:
1. Multiple Table Processing:
    - Processes multiple Bronze tables to corresponding Silver tables using configurable rules for each table.
    - Supports incremental data processing by comparing timestamps with previously processed records.
    - Supports partitioning for improved performance, especially on large datasets.

2. Data Quality and Validation:
    - Applies data quality checks, including regex validation and categorical value enforcement.
    - Deduplicates records to ensure only unique rows are present in the Silver layer.
    - Isolates and handles dirty data by appending it to quarantine tables for further analysis.

3. Metrics and Logging:
    - Tracks metrics for each table processed, including rows read, processed, and deduplicated.
    - Logs detailed information for each step of the pipeline, including success, failure, and any issues encountered.
    - Supports comprehensive error handling with clear error messages for debugging.

4. ETL Execution and Parallelism:
    - Executes the pipeline for each table sequentially or in parallel, based on configurations.
    - Ensures efficient memory and resource management through dynamic partitioning and shuffle operations.

5. Performance Optimization:
    - Includes performance optimizations, such as dynamic partitioning, to handle large datasets efficiently.
    - Allows Z-Ordering on relevant columns to improve query performance and partition pruning on the Silver tables.

6. Serverless Compatibility:
    - Designed to run efficiently on Databricks serverless clusters, ensuring scalability and cost-efficiency.

Best Practices / Notes:
- Ensure that the `superstore_silver_config.yaml` file is correctly configured for each environment (e.g., development, production).
- Monitor the ETL metrics table to track pipeline performance and handle any anomalies.
- Use proper column mappings and ensure that metadata columns are included for effective data governance.
- Regularly check the audit and quarantine tables for any data anomalies or issues that need to be addressed.
- Ensure that the tables in the Silver layer are optimized for performance using techniques like Z-Ordering after large inserts or merges.

==============================================================
"""

# -------------------------------
# Core Python and PySpark Imports
# -------------------------------
import os
import sys
from pyspark.sql import SparkSession

# -------------------------------
# Spark Session Initialization
# -------------------------------
# Create Spark session (or reuse existing one)
spark = SparkSession.builder.getOrCreate()

# Set session timezone to UTC for consistent timestamp handling across layers
spark.conf.set("spark.sql.session.timeZone", "UTC")

# -------------------------------
# Environment Variables Setup
# -------------------------------
# Fetch runtime environment details from Databricks job context
env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

# Set environment variables for downstream modules (logging, configs, audit)
os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Add Shared Code Paths
# -------------------------------
# this notebook lives at <bundle_root>/superstore_orchestrator/layer_orchestrator/<name>
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
# Include shared utilities (logging, config, platform functions)
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")
# Include Silver layer specific transformation modules
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_silver")   


# -------------------------------
# Core Imports (Framework + Platform)
# -------------------------------
from superstore_logger import get_superstore_logger, log_event  # centralized logging framework
from superstore_orchestrator_execution_context import get_execution_context  # runtime run IDs
from superstore_platform_constants import SILVER_LAYER  # layer constant for consistency
from superstore_platform_config import (
    table,
    get_bronze_schema,
    get_silver_schema,
    get_quarantine_schema,
    get_audit_schema,
    get_metrics_schema
)
from superstore_backfill_utils import get_backfill_config, validate_backfill_impact # Import backfill utilities

# -------------------------------
# Execution Context (Run Tracking)
# -------------------------------
# Extract master and layer-level run identifiers for auditability
execution_context = get_execution_context(dbutils, allow_standalone=True)
master_run_id = execution_context["master_run_id"]
layer_run_id = execution_context["layer_run_id"]

# -------------------------------
# Reload Silver Transformation Module
# -------------------------------
# Ensures latest code changes are reflected in notebook execution
import importlib
import superstore_silver_module
importlib.reload(superstore_silver_module)

from superstore_silver_module import (
    bronze_to_silver_prod,   # core bronze → silver transformation logic
    write_etl_metrics        # Silver layer metrics collector
)

# -------------------------------
# Load Configuration (YAML)
# -------------------------------
import yaml

config_path = f"{BUNDLE_ROOT}/configs/superstore_silver_config/superstore_silver_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Extract Silver table configurations (entity-level processing definitions)
silver_table_configs_raw = config["silver_table_configs"]

# -------------------------------
# Initialize Logger
# -------------------------------
logger_silver = get_superstore_logger("superstore_silver_layer_orchestrator")

log_event(
    logger_silver,
    "INFO",
    "Superstore Silver layer ETL orchestrator started",
    master_run_id=master_run_id,
    layer_run_id=layer_run_id,
    layer=SILVER_LAYER
)

# -------------------------------
# Dynamically resolve table paths per environment
# -------------------------------
silver_table_configs = []
for cfg in silver_table_configs_raw:
    silver_table_configs.append({
        "bronze_table": table(get_bronze_schema(), cfg["bronze_table"].split('.')[-1]),
        "silver_table": table(get_silver_schema(), cfg["silver_table"].split('.')[-1]),
        "quarantine_table": table(get_quarantine_schema(), cfg["quarantine_table"].split('.')[-1]),
        "audit_table": table(get_audit_schema(), cfg["audit_table"].split('.')[-1]),
        "read_from_table": table(get_bronze_schema(), cfg["read_from_table"].split('.')[-1]),
        "business_keys": cfg["business_keys"],
        "business_columns": cfg["business_columns"],
        "meta_columns": cfg["meta_columns"],
        "numeric_cast_cols": cfg.get("numeric_cast_cols", {}),
        "regex_cols": cfg.get("regex_cols", {}),
        "categorical_allowed_vals": cfg.get("categorical_allowed_vals", {}),
        "metrics_table": table(get_metrics_schema(), cfg["metrics_table"].split('.')[-1]),
        "layer_name": cfg.get("layer_name", "silver"),
        "shuffle_partitions": cfg.get("shuffle_partitions", 200)
    })

# -------------------------------
# Bronze-Silver ETL wrapper
# -------------------------------
def process_multiple_tables(
    spark, silver_table_configs: list, master_run_id: str
):
    log_event(
        logger_silver,
        "INFO",
        "Starting to process multiple Bronze layer table rows to Silver tables",
        total_tables=len(silver_table_configs),
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=SILVER_LAYER
    )

    # Get backfill configuration
    backfill_config = get_backfill_config(
        dbutils,
        allow_full_refresh=False,  # Set True only if you want to enable full refresh
        max_days=365  # Maximum backfill range (1 year)
    )

    log_event(
        logger_silver,
        "INFO",
        "Silver run configuration loaded",
        run_mode=backfill_config["mode"],
        dry_run=backfill_config["dry_run"],
        is_windowed=backfill_config["is_windowed"],
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=SILVER_LAYER
    )

    # -------------------------------
    # Dry run
    # -------------------------------
    # Report what this run would touch, then exit before any write. Every task
    # that writes must honour dry_run: a preview that writes to some layers
    # while the DAG reports success is worse than having no flag, because it
    # creates false confidence exactly when an operator is being careful.
    if backfill_config["dry_run"]:
        for cfg in silver_table_configs:
            impact = validate_backfill_impact(
                spark,
                cfg["bronze_table"],
                backfill_config,
                date_column="ingestion_date"
            )
            log_event(
                logger_silver,
                "INFO",
                f"Dry-run impact for {cfg['bronze_table']}",
                run_mode=backfill_config["mode"],
                impact=impact,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=SILVER_LAYER
            )

        dbutils.notebook.exit(
            "DRY_RUN_COMPLETED run_mode=" + backfill_config["mode"] + " layer=silver (no data written)"
        )

    for cfg in silver_table_configs:
        bronze_table = cfg["bronze_table"]
        log_event(
            logger_silver,
            "INFO",
            "Processing Bronze table",
            table=bronze_table,
            master_run_id=master_run_id,
            layer_run_id=layer_run_id,
            layer=SILVER_LAYER
        )

        try:
            metrics = bronze_to_silver_prod(
                spark,
                bronze_table=cfg["bronze_table"],
                silver_table=cfg["silver_table"],
                audit_table=cfg["audit_table"],
                business_keys=cfg["business_keys"],
                business_columns=cfg["business_columns"],
                meta_columns=cfg["meta_columns"],
                numeric_cast_cols=cfg["numeric_cast_cols"],
                regex_cols=cfg["regex_cols"],
                categorical_allowed_vals=cfg["categorical_allowed_vals"],
                quarantine_table=cfg["quarantine_table"],
                shuffle_partitions=cfg["shuffle_partitions"],
                layer_name=cfg["layer_name"],
                metrics_table=cfg["metrics_table"],
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                backfill_config=backfill_config  # backfil parameter
            )

            if metrics.get("metrics_table"):
                write_etl_metrics(
                    spark,
                    metrics_table=metrics["metrics_table"],
                    layer_run_id=metrics["layer_run_id"],
                    master_run_id=metrics["master_run_id"],
                    layer_name=metrics["layer"],
                    read_from_table=cfg["read_from_table"],
                    table_name=metrics["table_name"],
                    start_ts=metrics["start_ts"],
                    end_ts=metrics["end_ts"],
                    read_rows=metrics["read_rows"],
                    good_rows=metrics["good_rows"],
                    dirty_rows=metrics["dirty_rows"],
                    deduplicated_rows=metrics["deduplicated_rows"],
                    unchanged_rows=metrics["unchanged_rows"],
                    duplicate_rows=metrics["duplicate_rows"],
                    inserted_rows=metrics["inserted_rows"],
                    updated_rows=metrics["updated_rows"],
                    throughput_rows_per_sec=metrics["throughput_rows_per_sec"],
                    skew_ratio=metrics["skew_ratio"],
                    run_status=metrics["run_status"],
                    load_type=metrics["load_type"],
                    duration_secs=metrics["duration_secs"],
                    notes=metrics["notes"]
                )

            log_event(
                logger_silver,
                "INFO",
                "Completed ETL for table",
                table=bronze_table,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=SILVER_LAYER
            )

        except Exception as e:
            log_event(
                logger_silver,
                "ERROR",
                f"ETL failed for table: {bronze_table}",
                table=bronze_table,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=SILVER_LAYER,
                error=str(e)
            )

    log_event(
        logger_silver,
        "INFO",
        "Finished processing Bronze to Silver ETL for all tables",
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=SILVER_LAYER
    )

# -------------------------------
# Execute Bronze → Silver ETL for multiple tables
# -------------------------------
if __name__ == "__main__":
    process_multiple_tables(spark, silver_table_configs, master_run_id)
